# 02 - Cleaning

Applies the agreed cleaning rules to `data/processed/online_retail_raw.parquet` (output of notebook 01).
All logic lives in `src/clean_data.py`; this notebook runs it, checks it and writes the outputs.

| Output | Content | Use |
|---|---|---|
| `data/processed/transactions_clean.parquet` | all remaining rows with flags | revenue |
| `data/processed/customer_transactions.parquet` | `has_customer & is_product` | cohort / RFM / LTV |
| `reports/cleaning_log.md` | rows per step + revenue summary | documentation |

In [1]:
import numbers
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.load_data import load_processed
from src import clean_data as cd

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 90)

## 1. Load raw parquet and run the cleaning steps

In [2]:
raw = load_processed()
result = cd.run_cleaning(raw)
df, log = result.df, result.log
print(f"Raw rows: {len(raw):,}  ->  clean rows: {len(df):,}  ({len(raw) - len(df):,} removed)")
log

Raw rows: 1,067,371  ->  clean rows: 1,027,016  (40,355 removed)


,step,rule,rows before,rows removed,rows after
0,1,Drop sheet overlap: 'Year 2010-2011' rows with InvoiceDate < 2010-12-10,1067371,22523,1044848
1,2,Drop exact duplicate rows (all columns except source_sheet),1044848,11812,1033036
2,3,Uppercase and strip StockCode,1033036,0,1033036
3,4,Drop Quantity < 0 where Invoice does not start with 'C' (stock adjustments),1033036,3393,1029643
4,5a,Drop Price <= 0,1029643,2626,1027017
5,5b,Drop invoices starting with 'A' (bad debt adjustments),1027017,1,1027016
6,6,"Add flags is_cancelled, has_customer, is_product",1027016,0,1027016
7,7,Description = most frequent Description per StockCode (fills missing),1027016,0,1027016
8,8,"Add line_revenue, invoice_month, is_partial_month",1027016,0,1027016
9,9,Flag is_outlier: |Quantity| > 99.9th percentile (not removed),1027016,0,1027016


## 2. Sanity checks

In [3]:
checks = {
    "No 2010-2011 sheet rows before 2010-12-10":
        not ((df["source_sheet"] == cd.OVERLAP_SHEET) & (df["InvoiceDate"] < cd.OVERLAP_END)).any(),
    "No exact duplicates (excl. source_sheet, description_raw kept as loaded)":
        not df.drop(columns=["source_sheet", "Description", "is_cancelled", "has_customer", "is_product",
                             "line_revenue", "invoice_month", "is_partial_month", "is_outlier"]).duplicated().any(),
    "StockCode upper-case and stripped": (df["StockCode"] == df["StockCode"].str.strip().str.upper()).all(),
    "No negative quantity outside cancellations": not ((df["Quantity"] < 0) & ~df["is_cancelled"]).any(),
    "No Price <= 0": (df["Price"] > 0).all(),
    "No 'A' invoices": not df["Invoice"].str.startswith("A").any(),
    "No missing Description": df["Description"].notna().all(),
    "line_revenue == Quantity * Price": (df["line_revenue"] == df["Quantity"] * df["Price"]).all(),
}
for name, ok in checks.items():
    print(f"{'OK ' if ok else 'FAIL'}  {name}")
assert all(checks.values())

OK   No 2010-2011 sheet rows before 2010-12-10
OK   No exact duplicates (excl. source_sheet, description_raw kept as loaded)
OK   StockCode upper-case and stripped
OK   No negative quantity outside cancellations
OK   No Price <= 0
OK   No 'A' invoices
OK   No missing Description
OK   line_revenue == Quantity * Price


## 3. Flags

In [4]:
flag_cols = ["is_cancelled", "has_customer", "is_product", "is_partial_month", "is_outlier"]
pd.DataFrame({"True": df[flag_cols].sum(), "pct": (df[flag_cols].mean() * 100).round(2)})

,True,pct
is_cancelled,19104,1.86
has_customer,797815,77.68
is_product,1021272,99.44
is_partial_month,25193,2.45
is_outlier,1002,0.10


### is_product

Non-product = `NON_PRODUCT_CODES` or codes starting with GIFT / TEST. DCGS codes, PADS and SP1002 count as products.

In [5]:
print("Non-product codes (rows):")
print(df.loc[~df["is_product"], "StockCode"].value_counts().to_string())
print("\nProduct codes that are not 5 digits + optional letters:")
print(sorted(df.loc[df["is_product"] & ~df["StockCode"].str.match(r"^\d{5}"), "StockCode"].unique()))

Non-product codes (rows):
StockCode
POST            2079
DOT             1418
M               1385
C2               274
D                173
S                101
BANK CHARGES     100
ADJUST            67
AMAZONFEE         36
GIFT_0001_20      26
GIFT_0001_30      24
CRUK              16
GIFT_0001_10      14
TEST001           13
GIFT_0001_50       6
GIFT_0001_40       5
ADJUST2            3
GIFT_0001_80       2
TEST002            1
GIFT_0001_70       1

Product codes that are not 5 digits + optional letters:
['DCGS0003', 'DCGS0004', 'DCGS0037', 'DCGS0041', 'DCGS0044', 'DCGS0058', 'DCGS0062', 'DCGS0066N', 'DCGS0068', 'DCGS0069', 'DCGS0070', 'DCGS0072', 'DCGS0075', 'DCGS0076', 'DCGSSBOY', 'DCGSSGIRL', 'PADS', 'SP1002']


### Description standardisation

Missing descriptions were all on zero-price rows, so they are already gone by step 7. The step mostly
removes whitespace variants, but it also unifies wording that changed over time. `description_raw` keeps the original.

In [6]:
stripped_raw = df["description_raw"].str.strip()
whitespace_only = ((df["description_raw"] != df["Description"]) & (stripped_raw == df["Description"])).sum()
text_changed = stripped_raw != df["Description"]
print(f"Missing before step 7: {result.notes['descriptions_missing_before']:,}, after: {result.notes['descriptions_missing_after']:,}")
print(f"Rows changed only by whitespace: {whitespace_only:,}")
print(f"Rows with different wording: {text_changed.sum():,} across {df.loc[text_changed, 'StockCode'].nunique():,} StockCodes")
df.loc[text_changed, ["StockCode", "description_raw", "Description"]].drop_duplicates().sample(15, random_state=0)

Missing before step 7: 0, after: 0
Rows changed only by whitespace: 192,108
Rows with different wording: 60,401 across 601 StockCodes


,StockCode,description_raw,Description
726953,23196,RETRO LEAVES MAGNETIC NOTEPAD,VINTAGE LEAF MAGNETIC NOTEPAD
1654,21243,PINK SPOTTY PLATE,PINK POLKADOT PLATE
110,22029,SPACE BOY BIRTHDAY CARD,SPACEBOY BIRTHDAY CARD
705556,22986,WRAP GINGHAM ROSE,GINGHAM ROSE WRAP
357388,21039,RED RETROSPOT SHOPPING BAG,RED SPOTTY SHOPPING BAG
116618,22587,"FELTCRAFT HAIRBAND, S/2, RED/BLUE",FELTCRAFT HAIRBAND RED AND BLUE
278262,22641,SET OF 4 NAPKIN CHARMS INSTUMENT,SET OF 4 NAPKIN CHARMS INSTRUMENT
905463,23413,DECORATIVE VINTAGE COFFEE BOX,VINTAGE COFFEE GRINDER BOX
346243,20658,RED RETROSPOT LUGGAGE TAG,RED SPOTTY LUGGAGE TAG
660455,23283,DOORMAT VINTAGE LEAVES DESIGN,DOORMAT VINTAGE LEAF


### Outliers (flagged, not removed)

In [7]:
print(f"Threshold |Quantity| > {result.notes['outlier_threshold']:,.0f}: {df['is_outlier'].sum():,} rows, "
      f"£{df.loc[df['is_outlier'], 'line_revenue'].sum():,.0f} net revenue")
df[df["is_outlier"]].nlargest(10, "Quantity")[["Invoice", "StockCode", "Description", "Quantity", "Price",
                                                 "line_revenue", "Customer ID", "InvoiceDate"]]

Threshold |Quantity| > 504: 1,002 rows, £714,365 net revenue


,Invoice,StockCode,Description,Quantity,Price,line_revenue,Customer ID,InvoiceDate
1025534,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,168469.6,16446,2011-12-09 09:15:00
553565,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,77183.6,12346,2011-01-18 10:01:00
89017,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,0.10,1915.2,13902,2010-02-15 11:57:00
124538,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,0.10,1296.0,13902,2010-03-17 13:09:00
124540,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,0.10,1296.0,13902,2010-03-17 13:09:00
124541,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,0.10,1274.4,13902,2010-03-17 13:09:00
124539,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,0.10,1248.0,13902,2010-03-17 13:09:00
132228,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,0.25,2500.0,17940,2010-03-23 15:36:00
132229,502269,21982,PACK OF 12 SUKI TISSUES,10000,0.25,2500.0,17940,2010-03-23 15:36:00
132230,502269,21980,PACK OF 12 RED RETROSPOT TISSUES,10000,0.25,2500.0,17940,2010-03-23 15:36:00


## 4. Revenue summary

In [8]:
summary = cd.revenue_summary(df)
summary

,Metric,GBP,Share
0,Gross revenue (excl. cancellations),"20,465,198",100.0% of gross
1,Cancellations,"-1,462,051",-7.1% of gross
2,Net revenue,"19,003,148",100.0% of net
3,"Products, net","18,927,399",99.6% of net
4,"Postage (POST, DOT, C2, C3), net","433,411",2.3% of net
5,"Fees (AMAZONFEE, BANK CHARGES, CRUK), net","-264,936",-1.4% of net
6,"Other non-product (M, D, S, ADJUST, vouchers, TEST), net","-92,726",-0.5% of net
7,"Revenue with no Customer ID, gross","3,090,394",15.1% of gross
8,"Revenue with no Customer ID, net","2,713,156",14.3% of net


## 5. Save outputs

In [9]:
customers = cd.customer_subset(df)
p1 = cd.save(df, cd.CLEAN_PATH)
p2 = cd.save(customers, cd.CUSTOMER_PATH)
print(f"{p1.name}: {len(df):,} rows")
print(f"{p2.name}: {len(customers):,} rows, {customers['Customer ID'].nunique():,} customers "
      f"({customers['is_cancelled'].sum():,} cancellation rows kept)")

transactions_clean.parquet: 1,027,016 rows
customer_transactions.parquet: 794,183 rows, 5,875 customers (17,587 cancellation rows kept)


In [10]:
def md_table(frame: pd.DataFrame) -> str:
    cols = list(frame.columns)
    lines = ["| " + " | ".join(map(str, cols)) + " |", "|" + "---|" * len(cols)]
    lines += ["| " + " | ".join(f"{v:,}" if isinstance(v, numbers.Integral) and not isinstance(v, bool) else str(v).replace("|", "\|") for v in row) + " |"
              for row in frame.itertuples(index=False)]
    return "\n".join(lines)

notes = [
    f"- Raw rows: {len(raw):,}. Clean rows: {len(df):,}. Removed: {len(raw) - len(df):,} ({(len(raw) - len(df)) / len(raw):.2%}).",
    f"- `customer_transactions.parquet`: {len(customers):,} rows, {customers['Customer ID'].nunique():,} customers. Cancellations are kept as negative rows.",
    f"- Step 7: no descriptions were missing any more (all were on Price = 0 rows). "
    f"{whitespace_only:,} rows changed only by whitespace, {text_changed.sum():,} rows got different wording. Original kept in `description_raw`.",
    f"- Step 9: threshold |Quantity| > {result.notes['outlier_threshold']:,.0f}, {df['is_outlier'].sum():,} rows flagged.",
    "- `is_partial_month` marks December 2011 (data ends 2011-12-09).",
    "- Non-product codes: " + ", ".join(sorted(cd.NON_PRODUCT_CODES)) + ", plus codes starting with GIFT or TEST. DCGS codes, PADS and SP1002 count as products.",
]
parts = [
    "# Cleaning Log: Online Retail II",
    f"Generated by `notebooks/02_cleaning.ipynb`. Logic: `src/clean_data.py`.",
    "## Steps",
    md_table(log),
    "## Notes",
    "\n".join(notes),
    "## Revenue summary (transactions_clean)",
    "Gross excludes cancellation rows. Net = gross + cancellations. Non-product groups are net, so fees are negative.",
    md_table(summary),
]
report = ROOT / "reports" / "cleaning_log.md"
report.write_text("\n\n".join(parts) + "\n", encoding="utf-8")
print(f"Wrote {report}")

Wrote E:\BI\mini\online-retail-analysis\reports\cleaning_log.md


<>:4: SyntaxWarning: invalid escape sequence '\|'
<>:4: SyntaxWarning: invalid escape sequence '\|'
C:\Users\atefe\AppData\Local\Temp\ipykernel_5260\374187218.py:4: SyntaxWarning: invalid escape sequence '\|'
  lines += ["| " + " | ".join(f"{v:,}" if isinstance(v, numbers.Integral) and not isinstance(v, bool) else str(v).replace("|", "\|") for v in row) + " |"
